(projections)=
# Projection Images

```{toctree}
:maxdepth: 2
```

Suite2p computes several reference images during registration and detection. These images are used for quality assessment and as inputs to Cellpose anatomical segmentation.

## Overview

| Image | Source | Cellpose Mode |
|-------|--------|---------------|
| `meanImg` | Mean of registered movie | `anatomical_only=2` |
| `meanImgE` | Enhanced mean (HP filtered) | `anatomical_only=3` |
| `max_proj` | Max of HP-filtered movie | `anatomical_only=4` |

## Anatomical Modes

```{figure} _images/projections/01_anatomical_modes.png
:alt: Anatomical Modes
:name: proj-fig-anatomical
:width: 100%

Comparison of anatomical detection modes. **meanImg**: temporal mean of registered movie. **meanImgE**: enhanced mean with high-pass filtering. **max_proj**: maximum projection of HP-filtered movie.
```

```python
# meanImg: simple temporal average
meanImg = registered_movie.mean(axis=0)

# meanImgE: enhanced mean with local median subtraction
from scipy.ndimage import median_filter
Imed = median_filter(meanImg, size=d)
meanImgE = (meanImg - Imed) / median_filter(np.abs(meanImg - Imed), size=d)

# max_proj: max of temporally high-pass filtered movie
max_proj = hp_filtered_movie.max(axis=0)
```

## Spatial High-Pass Filter

The `spatial_hp_cp` parameter applies additional high-pass filtering before Cellpose segmentation.

```{figure} _images/projections/02_spatial_hp_filter.png
:alt: Spatial HP Filter
:name: proj-fig-hp
:width: 100%

Effect of `spatial_hp_cp` values on the max projection. Higher values sharpen cell boundaries but may amplify noise.
```

```python
from scipy.ndimage import gaussian_filter

def apply_hp_filter(img, diameter, spatial_hp_cp):
    sigma = diameter * spatial_hp_cp
    return img - gaussian_filter(img, sigma)
```

## Recommendations

```python
ops = {
    "anatomical_only": 3,      # use enhanced mean
    "spatial_hp_cp": 0,        # usually not needed with meanImgE
    "diameter": 6,             # cell size in pixels
    "cellprob_threshold": 0.0,
    "flow_threshold": 0.4,
}
```

If detection is poor:
- Try `anatomical_only=4` (max projection)
- Increase `spatial_hp_cp` to 1-3
- Adjust `diameter` to match cell sizes

## See Also

- {doc}`User Guide <user_guide>` - Complete parameter reference
- [Cellpose Documentation](https://cellpose.readthedocs.io/) - Cellpose model details